In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_inventory_kpis
# PURPOSE   : gold.fact_inventory_full → gold.fact_inventory_kpis
# LAYER     : Gold (rolling inventory KPIs)
# FREQUENCY : Daily (after gold_fact_inventory_full completes)
# WRITE     : Overwrite partitioned by snapshot_date (dynamic)
# GRAIN     : store_id + product_id + snapshot_date
#
# LOGIC: Reads fact_inventory_full, adds rolling 30d and 90d KPIs:
#   inventory_turnover_ratio, overstock_risk_index, dead stock detection,
#   avg days on hand, stock value at risk, inventory_health classification
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import GOLD_FACT_INVENTORY_KPIS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col, round, when,
    avg, sum as spark_sum
)
from pyspark.sql.window import Window
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["gold_fact_inventory_full"]
TARGET_TABLE = cfg["tables"]["gold_fact_inventory_kpis"]
PIPELINE     = "gold_fact_inventory_kpis"

In [0]:
# ── Read + Compute KPIs + Write ──────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    fact = spark.read.table(SOURCE_TABLE)
    rows_read = fact.count()
    print(f"[READ] {rows_read} rows from {SOURCE_TABLE}")
 
    # Derive stock_value — not stored in fact_inventory_full, computed here
    fact = fact.withColumn("stock_value",
        col("current_stock_qty") * col("cost_price"))
 
    w30 = (Window.partitionBy("store_id", "product_id")
                 .orderBy("snapshot_date").rowsBetween(-30, 0))
    w90 = (Window.partitionBy("store_id", "product_id")
                 .orderBy("snapshot_date").rowsBetween(-90, 0))
 
    df = (
        fact
 
        # KPI 1: Inventory Turnover Ratio = COGS 30d / avg stock value 30d
        .withColumn("cogs_30d",
            spark_sum(col("units_sold") * col("cost_price")).over(w30))
        .withColumn("avg_stock_value_30d",
            round(avg("stock_value").over(w30), 2))
        .withColumn("inventory_turnover_ratio",
            round(
                when(col("avg_stock_value_30d") > 0,
                     col("cogs_30d") / col("avg_stock_value_30d"))
                .otherwise(lit(0)), 3))
 
        # KPI 2: Overstock Risk Index = current_stock / (30d avg daily sales × 30)
        .withColumn("avg_daily_sales_30d",
            round(avg("units_sold").over(w30), 2))
        .withColumn("overstock_risk_index",
            round(
                when(col("avg_daily_sales_30d") > 0,
                     col("current_stock_qty") /
                     (col("avg_daily_sales_30d") * 30))
                .otherwise(lit(999)), 2))
 
        # KPI 3: Dead Stock — zero sales in 90 days but stock > 0
        .withColumn("units_sold_90d",
            spark_sum("units_sold").over(w90))
        .withColumn("is_dead_stock",
            (col("units_sold_90d") == 0) & (col("current_stock_qty") > 0))
 
        # KPI 4: Avg days on hand (30d rolling, using stock_cover_days from fact)
        .withColumn("avg_days_on_hand_30d",
            round(avg(
                when(col("stock_cover_days").isNotNull(),
                     col("stock_cover_days"))
            ).over(w30), 1))
 
        # KPI 5: Stock Value at Risk — dead or overstock products
        .withColumn("stock_value_at_risk",
            round(
                when(col("is_dead_stock") | (col("overstock_risk_index") > 1.5),
                     col("stock_value"))
                .otherwise(lit(0)), 2))
 
        # Health classification
        .withColumn("inventory_health",
            when(col("stockout_flag") == 1,          "Critical — Stockout")
            .when(col("is_dead_stock"),               "Critical — Dead Stock")
            .when(col("overstock_risk_index") > 1.5,  "Warning — Overstock")
            .when(col("reorder_flag") == 1,           "Warning — Low Stock")
            .otherwise("Healthy"))
 
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
 
        .select([f.name for f in GOLD_FACT_INVENTORY_KPIS_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("snapshot_date")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE).groupBy("inventory_health").count().show()